In [5]:
# Updated plotting + SIZR fitting for new CSV: populacja6_z_I.csv
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

# Resolve CSV path robustly
workspace = Path("/").joinpath(*Path.cwd().parts[:Path.cwd().parts.index("zombie-sim")+1]) if "zombie-sim" in Path.cwd().parts else Path.cwd()
candidates = [
    workspace / "populacja6_z_I.csv",
    workspace / "notebooks" / "populacja6_z_I.csv",
    Path("populacja6_z_I.csv"),
]
csv_path = None
for p in candidates:
    if p.exists():
        csv_path = p
        break
if csv_path is None:
    raise FileNotFoundError("Could not locate populacja6_z_I.csv in workspace or notebooks/")

# Load data
df = pd.read_csv(csv_path)
# Normalize expected column names
time_col = None
for c in df.columns:
    if c.lower() in ["time", "t"]:
        time_col = c
        break
if time_col is None:
    raise ValueError("CSV must contain a time column (time or t)")

# Expected series
cols = {}
for key in ["S", "I", "Z", "R"]:
    match = None
    for c in df.columns:
        if c.lower() == key.lower():
            match = c
            break
    if match is None:
        raise ValueError(f"Missing '{key}' column in CSV")
    cols[key] = match

# Prepare output dirs
fig_dir = workspace / "fig"
res_dir = workspace / "results"
fig_dir.mkdir(parents=True, exist_ok=True)
res_dir.mkdir(parents=True, exist_ok=True)

# Palette
palette = {"S": "#1f77b4", "I": "#ff7f0e", "Z": "#2ca02c", "R": "#7f7f7f"}

# Line plot
plot_df = df[[time_col, cols["S"], cols["I"], cols["Z"], cols["R"]]].rename(
    columns={cols["S"]: "S", cols["I"]: "I", cols["Z"]: "Z", cols["R"]: "R"}
)
plt.figure(figsize=(10, 4.8))
for col in ["S", "I", "Z", "R"]:
    sns.lineplot(data=plot_df, x=time_col, y=col, label=col, color=palette[col])
plt.title("Populacje w czasie — linie (S, I, Z, R)")
plt.xlabel("Czas")
plt.ylabel("Liczebność")
plt.legend()
fig_path1 = fig_dir / "populations_line.png"
plt.savefig(fig_path1, bbox_inches="tight")
plt.close()

# Stacked area
plt.figure(figsize=(10, 4.8))
plt.stackplot(plot_df[time_col], plot_df["S"], plot_df["I"], plot_df["Z"], plot_df["R"],
              labels=["S", "I", "Z", "R"], colors=[palette["S"], palette["I"], palette["Z"], palette["R"]])
plt.title("Populacje w czasie — stacked area (S, I, Z, R)")
plt.xlabel("Czas")
plt.ylabel("Liczebność")
plt.legend(loc="upper left")
fig_path2 = fig_dir / "populations_stacked.png"
plt.savefig(fig_path2, bbox_inches="tight")
plt.close()

# Percent stacked
total = plot_df[["S", "I", "Z", "R"]].sum(axis=1).replace(0, np.nan)
normalized = plot_df[["S", "I", "Z", "R"]].div(total, axis=0)
plt.figure(figsize=(10, 4.8))
plt.stackplot(plot_df[time_col], normalized["S"], normalized["I"], normalized["Z"], normalized["R"],
              labels=["S", "I", "Z", "R"], colors=[palette["S"], palette["I"], palette["Z"], palette["R"]])
plt.title("Populacje w czasie — udział procentowy (S, I, Z, R)")
plt.xlabel("Czas")
plt.ylabel("Udział [%]")
plt.legend(loc="upper left")
fig_path3 = fig_dir / "populations_percent.png"
plt.savefig(fig_path3, bbox_inches="tight")
plt.close()

# Rolling mean
win = 5
smoothed = plot_df[["S", "I", "Z", "R"]].rolling(win, min_periods=1).mean()
plt.figure(figsize=(10, 4.8))
for col in ["S", "I", "Z", "R"]:
    sns.lineplot(x=plot_df[time_col], y=smoothed[col], label=f"{col} (rolling {win})", color=palette[col])
plt.title("Populacje w czasie — rolling mean (S, I, Z, R)")
plt.xlabel("Czas")
plt.ylabel("Liczebność (wygładzone)")
plt.legend()
fig_path4 = fig_dir / "populations_rolling.png"
plt.savefig(fig_path4, bbox_inches="tight")
plt.close()

# --- SIZR model fitting ---
# ODE system

def sizr_rhs(t, y, beta, delta, rho, gamma, Lambda):
    S, I, Z, R = y
    N = max(S + I + Z, 1e-8)
    dS = -beta * S * Z / N + delta * I + Lambda * S
    dI = beta * S * Z / N - rho * I - delta * I
    dZ = rho * I - gamma * Z
    dR = gamma * Z
    return [dS, dI, dZ, dR]

# Initial conditions from first row
S0 = float(plot_df["S"].iloc[0])
I0 = float(plot_df["I"].iloc[0])
Z0 = float(plot_df["Z"].iloc[0])
R0 = float(plot_df["R"].iloc[0])
y0 = [S0, I0, Z0, R0]

# Time grid
t = plot_df[time_col].to_numpy().astype(float)

# Simulation helper

def simulate(params):
    beta, delta, rho, gamma, Lambda = params
    sol = solve_ivp(lambda tt, yy: sizr_rhs(tt, yy, beta, delta, rho, gamma, Lambda),
                    (t[0], t[-1]), y0, t_eval=t, method="RK45")
    return sol.y  # shape (4, len(t))

# Residuals vs. data
Y_data = np.vstack([
    plot_df["S"].to_numpy(),
    plot_df["I"].to_numpy(),
    plot_df["Z"].to_numpy(),
    plot_df["R"].to_numpy(),
])

# Initial guess and bounds
x0 = np.array([0.05, 0.05, 0.05, 0.05, 0.0])  # beta, delta, rho, gamma, Lambda
lower = np.array([0.0, 0.0, 0.0, 0.0, 0.0])
upper = np.array([1.0, 1.0, 1.0, 1.0, 0.1])


def residuals(params):
    Y_pred = simulate(params)
    return (Y_pred - Y_data).ravel()

res = least_squares(residuals, x0, bounds=(lower, upper), verbose=0)
params_fit = {
    "beta": float(res.x[0]),
    "delta": float(res.x[1]),
    "rho": float(res.x[2]),
    "gamma": float(res.x[3]),
    "Lambda": float(res.x[4]),
    "cost": float(res.cost),
    "success": bool(res.success),
}

# Save params
params_path = res_dir / "sizr_params_populacja6_z_I.json"
with open(params_path, "w", encoding="utf-8") as f:
    json.dump(params_fit, f, ensure_ascii=False, indent=2)

# Plot fit overlay
Y_pred = simulate(res.x)
plt.figure(figsize=(10, 6))
for i, label in enumerate(["S", "I", "Z", "R"]):
    plt.subplot(2, 2, i+1)
    plt.plot(t, Y_data[i], label=f"{label} data", color=palette[label], linewidth=2)
    plt.plot(t, Y_pred[i], label=f"{label} fit", color=palette[label], linestyle="--")
    plt.title(label)
    plt.xlabel("Czas")
    plt.ylabel("Liczebność")
    plt.legend()
plt.tight_layout()
fit_fig_path = fig_dir / "sizr_fit.png"
plt.savefig(fit_fig_path, bbox_inches="tight")
plt.close()

print(str(fig_path1), str(fig_path2), str(fig_path3), str(fig_path4), str(fit_fig_path), str(params_path))

/home/rafal/Documents/GitHub/zombie-sim/fig/populations_line.png /home/rafal/Documents/GitHub/zombie-sim/fig/populations_stacked.png /home/rafal/Documents/GitHub/zombie-sim/fig/populations_percent.png /home/rafal/Documents/GitHub/zombie-sim/fig/populations_rolling.png /home/rafal/Documents/GitHub/zombie-sim/fig/sizr_fit.png /home/rafal/Documents/GitHub/zombie-sim/results/sizr_params_populacja6_z_I.json
